# HCT-QHead：三参数分位数预测

输入为 15°、30°、45° 地震，模型输出顺序为 **Dn、Ip、Is**。请从包含 `model.py` 和 `well/` 的 `Code` 目录启动 Notebook。

## 目录

1. [导入依赖与模型接口](#imports)
2. [参数配置与模型组装](#configuration)
3. [模型训练与验证](#training)
4. [井上测试与结果导出](#well-evaluation)
5. [剖面预测与结果导出](#profile-prediction)
6. [整个工区预测与结果导出](#workarea-prediction)

**运行顺序：** 首次训练按第 1 → 2 → 3 节执行，再按需运行第 4、5、6 节。已有匹配的模型权重和标准化器时，执行第 1、2 节后可直接运行第 4、5 或第 6 节。重新开始训练前，请重新执行第 2 节。

**重复运行：** 使用 conda 的 `torch` 环境；应用确定性设置后，请重启内核并从第 1 节开始执行。第 1 节在导入 PyTorch 前配置 cuBLAS，第 2 节启用严格确定性算法并固定随机种子。训练速度可能略有下降；重复结果的比较应保持相同的硬件、软件版本、数据和参数。


<a id="imports"></a>

## 1. 导入依赖与模型接口

导入统一组装接口 `build_model`，以及数据处理、训练、评价和绘图所需依赖。本节只导入接口，下一节根据参数创建模型。


In [ ]:
# Cell 1：导入模型组装入口与运行依赖。从本 Notebook 所在的项目根目录运行。
# 环境依赖：torch、numpy、pandas、scipy、scikit-learn、joblib、matplotlib、tqdm、ipython。
# GPU 确定性配置须在导入 torch、初始化 CUDA 之前设置。
# 修改后请重启 torch 环境的内核，再从第 1 节开始运行。
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import math
import random
import re
import time
from pathlib import Path

import joblib
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from matplotlib.ticker import MultipleLocator
from scipy.stats import pearsonr
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from model import build_model

<a id="configuration"></a>

## 2. 参数配置与模型组装

集中配置数据与输出路径、输入和标签顺序、井划分、随机种子、网络参数、损失参数、训练设置及剖面、工区预测选项。参数字典留空时使用默认值；填写相应键即可覆盖。

模型组合为 `hct + qhead + quantile`，默认预测分位数为 `0.025、0.5、0.975`。设置随机种子后完成模型和损失函数实例化。


In [ ]:
# Cell 2：参数配置与模型组装。重新开始训练时，先重新执行本 Cell。
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "model.py").is_file() or not (PROJECT_DIR / "well").is_dir():
    raise FileNotFoundError("请从包含 model.py 和 well/ 的项目根目录运行 Notebook。")
WELL_DIR = PROJECT_DIR / "well"
OUTPUT_DIR = PROJECT_DIR / "HCT-QHead"
CSV_PATTERN = "W*.csv"

# 参数留空时使用组件默认值；这里只填写需要覆盖的项。
BACKBONE = "hct"
HEAD = "qhead"
LOSS = "quantile"
BACKBONE_PARAMS = {}  # 默认 base_channels=16、embed_dim=256、nhead=4、depth=3。
HEAD_PARAMS = {}  # 默认 out_dim=3、dropout=0.2；输入维度自动匹配主干。
LOSS_PARAMS = {}  # 默认 target_weights=(2,1,1)、beta=0.1。

# 输入和标签顺序必须与数据、标准化器一致。
feature_columns = ["Seis15", "Seis30", "Seis45"]
target_columns = ["Dn", "Ip", "Is"]
required_columns = feature_columns + target_columns
train_wells = list(range(0, 15))
val_wells = list(range(15, 17))
test_wells = list(range(17, 20))
wells_to_plot = list(range(20))
well_roles = {
    **{i: "Train" for i in train_wells},
    **{i: "Val" for i in val_wells},
    **{i: "Test" for i in test_wells},
}

SEED = 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_EPOCHS = 1000
LEARNING_RATE = 4e-4
WEIGHT_DECAY = 1e-3
FACTOR = 0.9
PATIENCE = 10
MIN_LR = 5e-7
BATCH_SIZE = 1  # 变长井逐井训练，不补齐；当前保持为 1。
USE_EARLY_STOPPING = True
EARLY_STOP_PATIENCE = 100
TAU_DISCRETE_VALUES = [0.025, 0.5, 0.975]
SAMPLES_PER_STEP = 3
TAU_SAMPLING_STRATEGY = "discrete"  # discrete 或 continuous。
quantiles = TAU_DISCRETE_VALUES
if len(quantiles) != 3 or not 0 < quantiles[0] < quantiles[1] == 0.5 < quantiles[2] < 1:
    raise ValueError("井上评价需要递增的三个分位数，中间为 0.5。")
q_low, q_mid, q_high = quantiles

MODEL_SAVE_PATH = OUTPUT_DIR / "best_model.pt"
SCALER_X_PATH = OUTPUT_DIR / "scaler_X.pkl"
SCALER_Y_PATH = OUTPUT_DIR / "scaler_y.pkl"
output_root_folder = OUTPUT_DIR / "bywell"

# 剖面预测与 CSV 导出。
INPUT_ROOT = PROJECT_DIR / "section"
OUTPUT_ROOT = OUTPUT_DIR / "byprofile"
INPUT_ANGLES = [15, 30, 45]
TARGET_COLUMNS = target_columns
WEIGHT_PATH = MODEL_SAVE_PATH
DEVICE_TAG = "gpu" if DEVICE == "cuda" else "cpu"
CSV_FLOAT_FORMAT = "%.6f"
PROFILE_NAMES = None  # None 处理所有 profile_s*；也可设为 ["profile_s1"]。
QUANTILES_TO_PREDICT = TAU_DISCRETE_VALUES

# 整个工区预测（第 6 节）：与井、剖面的输出目录独立。
WORKAREA_INPUT_DIR = PROJECT_DIR / "workarea"
WORKAREA_OUTPUT_DIR = OUTPUT_DIR / "workarea"  # 运行时自动创建。
WORKAREA_ZERO_TOLERANCE = 0.0  # 任一角度为零或非有限值时，该采样点无效。
WORKAREA_EMPTY_OUTPUT_VALUE = 0.0  # 与参考工区代码一致，无效位置输出零。
WORKAREA_CSV_ROW_CHUNK_SIZE = 100  # 超宽 CSV 按时间行分块写出，降低内存占用。

# 绘图设置
mpl.rcParams["font.sans-serif"] = ["Noto Sans CJK JP", "DejaVu Sans"]
mpl.rcParams["axes.unicode_minus"] = False

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
# 同时约束注意力等 CUDA 运算；固定种子与 cuDNN 设置不足以保证全部运算确定。
# 严格模式不使用 warn_only=True，避免非确定性算子仅警告后继续执行。
torch.use_deterministic_algorithms(True)
if BATCH_SIZE != 1:
    raise ValueError("当前变长井训练使用 BATCH_SIZE=1。")
model, criterion = build_model(
    backbone=BACKBONE,
    head=HEAD,
    loss=LOSS,
    backbone_params=BACKBONE_PARAMS,
    head_params=HEAD_PARAMS,
    loss_params=LOSS_PARAMS,
)
model, criterion = model.to(DEVICE), criterion.to(DEVICE)
model_config = model.config
if model_config["backbone_params"]["input_dim"] != len(feature_columns):
    raise ValueError("主干 input_dim 与输入列数不一致。")
if model_config["head_params"]["out_dim"] != len(target_columns):
    raise ValueError("预测头 out_dim 与标签列数不一致。")
TARGET_LOSS_WEIGHTS = model_config["loss_params"]["target_weights"]
print("组件：", BACKBONE, HEAD, LOSS, "；设备：", DEVICE)
print("模型参数量：", sum(p.numel() for p in model.parameters()))
if not isinstance(SAMPLES_PER_STEP, int) or SAMPLES_PER_STEP < 1:
    raise ValueError("SAMPLES_PER_STEP 必须是正整数。")

<a id="training"></a>

## 3. 模型训练与验证

读取并筛选井数据，仅使用训练井拟合标准化器，执行训练与验证、学习率调整和早停。一次训练目标计算复用主干特征，默认重复采样 3 次分位数。

权重和标准化器写入 `HCT-QHead/`：`best_model.pt`、`scaler_X.pkl`、`scaler_y.pkl`。训练结束显示损失曲线。


In [ ]:
# Cell 3：训练与验证。参数和模型由 Cell 2 提供。


def training_loss(x, y):
    """一次提取主干特征，多次逐点采样 tau，平均目标加权损失。"""
    features = model.backbone(x)
    reference = y[..., :1]
    total = 0.0
    for _ in range(SAMPLES_PER_STEP):
        if TAU_SAMPLING_STRATEGY == "discrete":
            values = np.random.choice(
                TAU_DISCRETE_VALUES, size=reference.numel(), replace=True
            )
            tau = (
                torch.from_numpy(values)
                .to(device=y.device, dtype=y.dtype)
                .reshape_as(reference)
            )
        elif TAU_SAMPLING_STRATEGY == "continuous":
            tau = torch.rand_like(reference)
        else:
            raise ValueError("TAU_SAMPLING_STRATEGY 只能为 discrete 或 continuous。")
        total = total + criterion(model.forward_from_features(features, tau), y, tau)
    return total / SAMPLES_PER_STEP


def _well_name_from_filename(path) -> str:
    """从文件名提取井名，用于数据划分和结果显示。"""
    return Path(path).stem


def _well_sort_key(path):
    """提取井号的数值排序键，避免 W10 排在 W2 前面。"""
    name = _well_name_from_filename(path)
    if name.startswith("W") and name[1:].isdigit():
        return (0, int(name[1:]), name)
    return (1, name)


def _time_col(df):
    """查找以 time 开头的时间列，忽略大小写。"""
    for c in df.columns:
        if str(c).lower().startswith("time"):
            return c
    raise ValueError(f"time column not found. columns={list(df.columns)}")


def load_well_df(file):
    """读取井 CSV 并按时间排序，删除输入或目标含零值、非有限值的行。

    训练和井上评估使用筛选后的连续行序列，不保留被删除行的时间间隔。
    """
    df = pd.read_csv(file)
    tcol = _time_col(df)

    missing = [c for c in required_columns if c not in df.columns]
    if missing:
        raise ValueError(f"{file} missing columns: {missing}")

    numeric_columns = [tcol] + required_columns
    for col in numeric_columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    valid = np.isfinite(df[required_columns]).all(axis=1)
    valid &= (df[required_columns] != 0).all(axis=1)
    valid &= np.isfinite(df[tcol])

    df = (
        df.loc[valid, [tcol] + required_columns]
        .sort_values(tcol)
        .reset_index(drop=True)
    )

    if df.empty:
        raise ValueError(f"{file} has no valid rows after removing zero/NaN values")

    return df


def load_well(file):
    """按 feature_columns 和 target_columns 的顺序返回井上输入、目标数组。"""
    df = load_well_df(file)
    X = df[feature_columns].values.astype(np.float32)
    y = df[target_columns].values.astype(np.float32)
    return X, y


class EarlyStopping:
    """跟踪验证损失，在改善时保存权重，连续未改善达到 patience 时触发早停。"""

    def __init__(self, patience=100, verbose=False, delta=0.0, path="best_model.pt"):
        """初始化早停计数、最佳损失记录和权重保存路径。"""
        self.patience = patience
        self.verbose = verbose
        self.delta = delta
        self.path = path

        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.inf

        self.best_epoch = 0
        self.best_train_loss = 0.0
        self.best_val_loss = 0.0

    def __call__(self, val_loss, tr_loss, epoch, model):
        """更新本轮验证损失与早停状态，必要时保存当前模型。"""
        score = -val_loss

        if self.best_score is None or score > self.best_score + self.delta:
            self.best_score = score
            self._save(val_loss, tr_loss, epoch, model)
            self.counter = 0
        else:
            self.counter += 1
            if self.verbose and self.counter % 10 == 0:
                print(f"早停计数器: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True

    def _save(self, val_loss, tr_loss, epoch, model):
        """保存模型 state_dict，同时记录最佳轮次及其训练、验证损失。"""
        if self.verbose:
            print(
                f"验证集损失下降 ({self.val_loss_min:.6f} -> {val_loss:.6f})，保存模型。"
            )
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "model_config": model_config,
                "features": feature_columns,
                "targets": target_columns,
            },
            self.path,
        )
        self.val_loss_min = val_loss
        self.best_epoch = epoch
        self.best_train_loss = tr_loss
        self.best_val_loss = val_loss


if __name__ == "__main__":
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    all_files = sorted(WELL_DIR.glob(CSV_PATTERN), key=_well_sort_key)
    assert len(all_files) > max(
        train_wells + val_wells + test_wells
    ), f"{WELL_DIR} 里文件数量不足。"

    print(f"project dir: {PROJECT_DIR}")
    print(f"well dir: {WELL_DIR}")
    print(f"output dir: {OUTPUT_DIR}")
    print(f"feature columns: {feature_columns}")
    print(f"target columns: {target_columns}")

    print("\n发现以下井的数据(按井名排序，并完成 zero/NaN 过滤检查):")
    for i, f in enumerate(all_files):
        well_name = _well_name_from_filename(f)
        valid_rows = len(load_well_df(f))
        if i in train_wells:
            role = "Train"
        elif i in val_wells:
            role = "Val"
        elif i in test_wells:
            role = "Test"
        else:
            role = "Unused"
        print(
            f"[{i:02d}] {well_name} <- {f.name}; "
            f"role={role}; valid_samples={valid_rows}"
        )

    # 标准化统计量仅由训练井拟合，验证和测试井复用这些统计量。
    X_train_all = np.vstack([load_well(all_files[i])[0] for i in train_wells])
    y_train_all = np.vstack([load_well(all_files[i])[1] for i in train_wells])

    scaler_X = StandardScaler().fit(X_train_all)
    scaler_y = StandardScaler().fit(y_train_all)

    def pack(X, y):
        """复用训练集拟合的标准化器，返回形状为 (T, 3) 的输入和目标张量。"""
        return (
            torch.tensor(scaler_X.transform(X), dtype=torch.float32),
            torch.tensor(scaler_y.transform(y), dtype=torch.float32),
        )

    train_dataset = [pack(*load_well(all_files[i])) for i in train_wells]
    val_dataset = [pack(*load_well(all_files[i])) for i in val_wells]

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=FACTOR,
        patience=PATIENCE,
        min_lr=MIN_LR,
    )

    early_stopper = None
    if USE_EARLY_STOPPING:
        early_stopper = EarlyStopping(
            patience=EARLY_STOP_PATIENCE,
            verbose=True,
            path=MODEL_SAVE_PATH,
        )
        print(f"\n已启用早停策略，耐心值为 {EARLY_STOP_PATIENCE}。")
    else:
        print("\n未启用早停策略，将训练完整的 Epochs。")

    train_hist = []
    val_hist = []
    lr_hist = []

    print(
        f"\n开始训练 HCT-QHead 三参数模型 "
        f"(每步采样 {SAMPLES_PER_STEP} 次, 策略: '{TAU_SAMPLING_STRATEGY}')..."
    )
    print(f"tau discrete values: {TAU_DISCRETE_VALUES}")
    print(f"target loss weights: {TARGET_LOSS_WEIGHTS}  # order: Dn, Ip, Is")

    t0 = time.time()

    # 每轮先更新训练权重，再用验证损失调整学习率和早停状态。
    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        tr_loss_sum = 0.0
        tr_weight_sum = 0.0

        for X, y in train_loader:
            X = X.to(DEVICE)
            y = y.to(DEVICE)

            loss = training_loss(X, y)

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            optimizer.step()

            T_k = y.shape[1]
            tr_loss_sum += loss.item() * T_k
            tr_weight_sum += T_k

        tr = tr_loss_sum / max(tr_weight_sum, 1.0)

        model.eval()
        vl_loss_sum = 0.0
        vl_weight_sum = 0.0

        with torch.no_grad():
            for X, y in val_loader:
                X = X.to(DEVICE)
                y = y.to(DEVICE)

                loss = training_loss(X, y)

                T_k = y.shape[1]
                vl_loss_sum += loss.item() * T_k
                vl_weight_sum += T_k

        vl = vl_loss_sum / max(vl_weight_sum, 1.0)

        train_hist.append(tr)
        val_hist.append(vl)

        lr_before = optimizer.param_groups[0]["lr"]
        # 学习率调度以本轮验证损失为依据。
        scheduler.step(vl)
        lr_now = optimizer.param_groups[0]["lr"]
        lr_hist.append(lr_now)

        if lr_now < lr_before:
            print(f"学习率降低: epoch {epoch}: {lr_before:.2e} -> {lr_now:.2e}")

        if epoch % 10 == 0 or epoch == 1:
            print(
                f"Epoch[{epoch}/{NUM_EPOCHS}] "
                f"LR={lr_now:.1e} "
                f"Train={tr:.6f} Val={vl:.6f}"
            )

        if USE_EARLY_STOPPING:
            early_stopper(vl, tr, epoch, model)
            if early_stopper.early_stop:
                print("触发早停。")
                break

    if not USE_EARLY_STOPPING:
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "model_config": model_config,
                "features": feature_columns,
                "targets": target_columns,
            },
            MODEL_SAVE_PATH,
        )
        print(f"\n训练完成 {NUM_EPOCHS} 轮，模型已保存至 {MODEL_SAVE_PATH}")

    if MODEL_SAVE_PATH.exists():
        model.load_state_dict(
            torch.load(MODEL_SAVE_PATH, map_location=DEVICE, weights_only=True)[
                "model_state_dict"
            ]
        )

    joblib.dump(scaler_X, SCALER_X_PATH)
    joblib.dump(scaler_y, SCALER_Y_PATH)

    elapsed = time.time() - t0

    print("\n" + "=" * 80)
    print(f"训练完成，用时 {int(elapsed // 60)}分 {elapsed % 60:.2f}秒")
    if USE_EARLY_STOPPING and early_stopper is not None:
        print(
            f"最佳 Epoch @{early_stopper.best_epoch}: "
            f"Train={early_stopper.best_train_loss:.6f} "
            f"Val={early_stopper.best_val_loss:.6f}"
        )
    print(f"saved model: {MODEL_SAVE_PATH}")
    print(f"saved scalers: {SCALER_X_PATH}, {SCALER_Y_PATH}")
    print("=" * 80 + "\n")

    plt.figure(figsize=(9, 4.6))
    plt.plot(train_hist, label="训练损失")
    plt.plot(val_hist, label="验证损失")

    if (
        USE_EARLY_STOPPING
        and early_stopper is not None
        and early_stopper.best_epoch > 0
    ):
        plt.axvline(
            early_stopper.best_epoch - 1,
            color="r",
            ls="--",
            label=f"最佳 @{early_stopper.best_epoch}",
        )

    plt.title("Direct Dn/Ip/Is AVO HCT-QHead 模型损失曲线")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.grid(True, ls="--", alpha=0.4)
    plt.legend()
    plt.tight_layout()
    plt.show()

<a id="well-evaluation"></a>

## 4. 井上测试与结果导出

加载保存的模型和标准化器，分别评价训练井、验证井和盲井。计算中位数预测的 MARE、RMSRE、相关系数 PCC，以及区间指标 PICP、MPIW。

每口井预测 CSV、逐井与分组指标表、井曲线图保存到 `HCT-QHead/bywell/`。本节可以在已有权重时跳过训练直接运行。


In [ ]:
# Cell 4：井上测试。已有权重时执行 Cell 1、2 后可直接运行。


def well_name_from_path(path):
    """从 CSV 文件路径提取井名。"""
    return Path(path).stem


def well_sort_key(path):
    """提取井号的数值排序键，避免 W10 排在 W2 前面。"""
    name = well_name_from_path(path)
    if name.startswith("W") and name[1:].isdigit():
        return (0, int(name[1:]), name)
    return (1, name)


def time_col(df):
    """查找以 time 开头的时间列，忽略大小写。"""
    for c in df.columns:
        if str(c).lower().startswith("time"):
            return c
    raise ValueError(f"time column not found. columns={list(df.columns)}")


def load_well_df(file):
    """读取井 CSV 并按时间排序，删除输入或目标含零值、非有限值的行。

    训练和井上评估使用筛选后的连续行序列，不保留被删除行的时间间隔。
    """
    df = pd.read_csv(file)
    tcol = time_col(df)
    missing = [c for c in required_columns if c not in df.columns]
    if missing:
        raise ValueError(f"{file} missing columns: {missing}")

    numeric_columns = [tcol] + required_columns
    for col in numeric_columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    valid = np.isfinite(df[required_columns]).all(axis=1)
    valid &= (df[required_columns] != 0).all(axis=1)
    valid &= np.isfinite(df[tcol])
    df = (
        df.loc[valid, [tcol] + required_columns]
        .sort_values(tcol)
        .reset_index(drop=True)
    )
    if df.empty:
        raise ValueError(f"{file} has no valid rows after removing zero/NaN values")
    return df


def weighted_role_summary(per_well_rows, roles=("Train", "Val", "Test")):
    """按数据集分组，以各井样本数加权平均井级指标。

    RMSRE 和相关系数在井级先计算，再加权；不是合并所有样本后重新计算。
    """
    df = pd.DataFrame(per_well_rows)
    if df.empty:
        return pd.DataFrame(columns=["Role", "Samples"])
    out_rows = []
    for role in roles:
        sub = df[df["Role"] == role].copy()
        if sub.empty:
            continue
        w = sub["Samples"].astype(float).to_numpy()
        wsum = float(np.sum(w)) if np.sum(w) > 0 else 0.0
        row = {"Role": role, "Samples": int(wsum)}
        for col in sub.columns:
            if col in ("Role", "Samples", "Well_Name"):
                continue
            vals = pd.to_numeric(sub[col], errors="coerce").to_numpy(dtype=float)
            row[col] = (
                float(np.nansum(vals * w) / wsum)
                if wsum > 0
                else float(np.nanmean(vals))
            )
        out_rows.append(row)
    return pd.DataFrame(out_rows)


def metrics(y_true, y_pred, y_low=None, y_high=None, y_range=1.0):
    """计算中位数预测误差、相关系数及分位数区间指标。

    相对误差分母为 y_true + 1e-8；PICP 为覆盖比例，MPIW 除以给定的 y_range。
    """
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[valid]
    y_pred = y_pred[valid]

    eps = 1e-8
    denom = y_true + eps

    out = {
        "MARE": (
            float(np.mean(np.abs((y_pred - y_true) / denom))) if len(y_true) else np.nan
        ),
        "RMSRE": (
            float(np.sqrt(np.mean(((y_pred - y_true) / denom) ** 2)))
            if len(y_true)
            else np.nan
        ),
        "PCC": (
            float(pearsonr(y_true, y_pred)[0])
            if len(y_true) > 1 and np.nanstd(y_true) > 0 and np.nanstd(y_pred) > 0
            else np.nan
        ),
    }

    if y_low is not None and y_high is not None:
        y_low = np.asarray(y_low).ravel()[valid]
        y_high = np.asarray(y_high).ravel()[valid]
        out["PICP"] = (
            float(np.mean((y_true >= y_low) & (y_true <= y_high)))
            if len(y_true)
            else np.nan
        )
        out["MPIW"] = (
            float(np.mean(y_high - y_low)) / y_range if len(y_true) else np.nan
        )

    return out


output_root_folder.mkdir(parents=True, exist_ok=True)
for target in target_columns:
    (output_root_folder / target).mkdir(parents=True, exist_ok=True)

all_files = sorted(WELL_DIR.glob(CSV_PATTERN), key=well_sort_key)
assert len(all_files) > max(wells_to_plot), "not enough well CSV files"
scaler_X = joblib.load(SCALER_X_PATH)
scaler_y = joblib.load(SCALER_Y_PATH)

checkpoint = torch.load(MODEL_SAVE_PATH, map_location=DEVICE, weights_only=True)
if checkpoint["model_config"] != model_config:
    raise ValueError("权重中的模型配置与 Cell 2 不一致，请核对组件和参数。")
if checkpoint["features"] != feature_columns or checkpoint["targets"] != target_columns:
    raise ValueError("权重记录的输入或标签顺序与 Cell 2 不一致。")
model, _ = build_model(**checkpoint["model_config"])
model = model.to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model.eval()

print(f"loaded model: {MODEL_SAVE_PATH}")
print(f"feature columns: {feature_columns}")
print(f"target columns: {target_columns}")
print(f"output root: {output_root_folder}")

cache = []
total_time = 0.0
with torch.no_grad():
    for k, idx in enumerate(wells_to_plot):
        df = load_well_df(all_files[idx])
        t = df[time_col(df)].to_numpy()
        X = df[feature_columns].values.astype(np.float32)
        y_true = df[target_columns].values.astype(np.float32)
        Xs = scaler_X.transform(X)
        X_tensor = torch.tensor(Xs, dtype=torch.float32, device=DEVICE)[None]

        if DEVICE == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        # 同一口井只计算一次主干；Q 维与 quantiles 顺序一致。
        prediction_all = (
            model.predict_quantiles(X_tensor, quantiles).squeeze(0).cpu().numpy()
        )
        pred_std = {q: prediction_all[:, qi, :] for qi, q in enumerate(quantiles)}
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        total_time += time.perf_counter() - t0

        pred_low = scaler_y.inverse_transform(pred_std[q_low])
        pred_mid = scaler_y.inverse_transform(pred_std[q_mid])
        pred_high = scaler_y.inverse_transform(pred_std[q_high])

        cache.append(
            {
                "idx": idx,
                "k": k,
                "well": well_name_from_path(all_files[idx]),
                "role": well_roles.get(idx, "Unknown"),
                "t": t,
                "X": X,
                "y_true": y_true,
                "pred_low": pred_low,
                "pred_mid": pred_mid,
                "pred_high": pred_high,
                "samples": len(df),
            }
        )

print(f"predicted wells: {len(cache)}; pure inference time: {total_time:.6f} sec")

all_weighted_tables = []
for target_i, target in enumerate(target_columns):
    all_true_target = np.concatenate([item["y_true"][:, target_i] for item in cache])
    y_range_global = float(np.nanmax(all_true_target) - np.nanmin(all_true_target))
    if not np.isfinite(y_range_global) or y_range_global <= 0:
        y_range_global = 1.0

    rows = []
    target_folder = output_root_folder / target
    for item in cache:
        y = item["y_true"][:, target_i]
        lo = item["pred_low"][:, target_i]
        mid = item["pred_mid"][:, target_i]
        hi = item["pred_high"][:, target_i]
        m = metrics(y, mid, lo, hi, y_range_global)
        rows.append(
            {
                "Role": item["role"],
                "Well_Name": item["well"],
                "Samples": item["samples"],
                **m,
            }
        )

        result = {"Time": item["t"]}
        for feature_i, feature_name in enumerate(feature_columns):
            result[feature_name] = item["X"][:, feature_i]
        result.update(
            {
                f"{target}_True": y,
                f"{target}_Pred_q{q_low:g}": lo,
                f"{target}_Pred_q{q_mid:g}": mid,
                f"{target}_Pred_q{q_high:g}": hi,
                "dist_left": np.abs(y - lo),
                "dist_right": np.abs(hi - y),
            }
        )
        pd.DataFrame(result).to_csv(
            target_folder / f'{item["well"]}.csv',
            index=False,
            float_format="%.4f",
        )

    summary = pd.DataFrame(rows)
    summary_path = target_folder / "metrics_summary.csv"
    summary.to_csv(summary_path, index=False, float_format="%.4f")
    print()
    print("=" * 80)
    print(target)
    display(summary)

    combined_bywell = []
    for row_dict in rows:
        combined_bywell.append(
            {
                "Role": row_dict["Role"],
                "Well_Name": row_dict["Well_Name"],
                "Samples": row_dict["Samples"],
                "AVO MARE": row_dict["MARE"],
                "AVO RMSRE": row_dict["RMSRE"],
                "AVO PCC": row_dict["PCC"],
                "AVO PICP": row_dict["PICP"],
                "AVO MPIW": row_dict["MPIW"],
            }
        )

    weighted_wide = weighted_role_summary(combined_bywell)
    final_rows = []
    for i, role in enumerate(["Train", "Val", "Test"]):
        rd = weighted_wide[weighted_wide["Role"] == role]
        if rd.empty:
            samples, vals = 0, {}
        else:
            r = rd.iloc[0]
            samples, vals = int(r["Samples"]), r

        final_rows.append(
            {
                "Role": role,
                "Samples": samples,
                "MARE": f'{vals.get("AVO MARE", 0):.4f}' if not rd.empty else "",
                "RMSRE": f'{vals.get("AVO RMSRE", 0):.4f}' if not rd.empty else "",
                "PCC": f'{vals.get("AVO PCC", 0):.4f}' if not rd.empty else "",
                "PICP": f'{vals.get("AVO PICP", 0):.4f}' if not rd.empty else "",
                "MPIW": f'{vals.get("AVO MPIW", 0):.4f}' if not rd.empty else "",
            }
        )

    formatted = pd.DataFrame(
        final_rows, columns=["Role", "Samples", "MARE", "RMSRE", "PCC", "PICP", "MPIW"]
    )
    wcsv_path = target_folder / "metrics_summary_weighted_role.csv"
    formatted.to_csv(wcsv_path, index=False)
    all_weighted_tables.append(formatted.assign(Target=target))
    print()
    print(f"weighted metrics saved to: {wcsv_path}")
    display(formatted)

    ncols = 5
    nrows = math.ceil(len(cache) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.0 * ncols, 4.5 * nrows), dpi=180)
    axes = np.array(axes).reshape(-1)
    finite_vals = []
    for item in cache:
        finite_vals.extend(
            [
                item["y_true"][:, target_i],
                item["pred_low"][:, target_i],
                item["pred_mid"][:, target_i],
                item["pred_high"][:, target_i],
            ]
        )
    finite_vals = np.concatenate([v[np.isfinite(v)] for v in finite_vals])
    x_min, x_max = float(np.nanmin(finite_vals)), float(np.nanmax(finite_vals))

    for item in cache:
        ax = axes[item["k"]]
        tt = item["t"]
        y = item["y_true"][:, target_i]
        lo = item["pred_low"][:, target_i]
        mid = item["pred_mid"][:, target_i]
        hi = item["pred_high"][:, target_i]
        ax.plot(y, tt, color="black", lw=1.2, label="True")
        ax.plot(mid, tt, "--", color="red", lw=1.2, label="Median")
        ax.fill_betweenx(
            tt,
            lo,
            hi,
            color="pink",
            alpha=0.45,
            label=f"{100 * (q_high - q_low):g}% PI",
        )
        ax.set_title(f'{item["well"]} ({item["role"]})')
        ax.set_xlabel(target)
        ax.set_ylabel("Time(s)" if item["k"] % ncols == 0 else "")
        ax.set_ylim(tt.max(), tt.min())
        ax.set_xlim(x_min, x_max)
        ax.yaxis.set_major_locator(MultipleLocator(0.05))
        ax.grid(True, ls="--", alpha=0.4)
    for j in range(len(cache), nrows * ncols):
        fig.delaxes(axes[j])
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False)
    plt.tight_layout(rect=(0, 0, 1, 0.98))
    fig_path = target_folder / f"prediction_{target}.png"
    plt.savefig(fig_path, dpi=300)
    print(f"saved figure: {fig_path}")
    plt.show()

if all_weighted_tables:
    all_weighted = pd.concat(all_weighted_tables, ignore_index=True)
    all_weighted = all_weighted[
        ["Target", "Role", "Samples", "MARE", "RMSRE", "PCC", "PICP", "MPIW"]
    ]
    all_weighted_path = (
        output_root_folder / "metrics_summary_weighted_role_all_targets.csv"
    )
    all_weighted.to_csv(all_weighted_path, index=False)
    print()
    print(f"all-target weighted metrics saved to: {all_weighted_path}")
    display(all_weighted)

<a id="profile-prediction"></a>

## 5. 剖面预测与结果导出

读取 `section/profile_s*` 中的三角度地震，逐道预测并反标准化，保留时间轴、道顺序和无效位置。每条剖面输出 Dn、Ip、Is 的 low、median、upper，共 9 个预测 CSV。

预测矩阵和耗时记录保存到 `HCT-QHead/byprofile/`。可在第 2 节设置 `PROFILE_NAMES` 选择剖面；本节仅导出预测，不生成剖面图，也不执行全工区预测。


In [ ]:
# Cell 5：剖面逐道预测与 CSV 导出，不绘图；全工区预测见第 6 节。


def discover_profiles(input_root, input_angles):
    """按剖面编号收集满足本阶段文件要求的剖面。"""
    input_root = Path(input_root)

    if not input_root.exists():
        raise FileNotFoundError(f"输入根目录不存在：{input_root.resolve()}")

    if not input_root.is_dir():
        raise NotADirectoryError(f"输入路径不是文件夹：{input_root.resolve()}")

    profile_pattern = re.compile(
        r"^profile_s(\d+)$",
        flags=re.IGNORECASE,
    )

    valid_profiles = []
    skipped_profiles = []

    for profile_dir in input_root.iterdir():
        if not profile_dir.is_dir():
            continue

        match = profile_pattern.fullmatch(profile_dir.name)

        if match is None:
            continue

        profile_id = int(match.group(1))

        required_files = [
            profile_dir / f"seismic_theta{angle}.csv" for angle in input_angles
        ]

        missing_files = [path.name for path in required_files if not path.exists()]

        if missing_files:
            skipped_profiles.append(
                {
                    "profile_name": profile_dir.name,
                    "missing_files": missing_files,
                }
            )
            continue

        valid_profiles.append(
            {
                "profile_id": profile_id,
                "profile_name": profile_dir.name,
                "profile_dir": profile_dir,
            }
        )

    valid_profiles.sort(key=lambda item: item["profile_id"])

    if skipped_profiles:
        print("以下剖面因输入文件不完整而被跳过：")

        for item in skipped_profiles:
            print(
                f"  - {item['profile_name']}：缺少 " + ", ".join(item["missing_files"])
            )

    if not valid_profiles:
        required_names = [f"seismic_theta{angle}.csv" for angle in input_angles]

        raise FileNotFoundError(
            f"在 {input_root.resolve()} 下未找到有效剖面。\n"
            f"有效剖面文件夹名称应为 profile_s数字，"
            f"并同时包含：{required_names}"
        )

    return valid_profiles


PROFILES_TO_PROCESS = discover_profiles(
    input_root=INPUT_ROOT,
    input_angles=INPUT_ANGLES,
)
if PROFILE_NAMES is not None:
    found = {p["profile_name"]: p for p in PROFILES_TO_PROCESS}
    missing = set(PROFILE_NAMES) - set(found)
    if missing:
        raise FileNotFoundError(f"缺少可预测剖面：{sorted(missing)}")
    PROFILES_TO_PROCESS = [found[name] for name in PROFILE_NAMES]

print("\n自动识别到的有效剖面：")
for profile_info in PROFILES_TO_PROCESS:
    print(f"  - {profile_info['profile_name']}")

print(f"共识别到 {len(PROFILES_TO_PROCESS)} 条有效剖面。")

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

required_model_files = [
    WEIGHT_PATH,
    SCALER_X_PATH,
    SCALER_Y_PATH,
]

missing_model_files = [str(path) for path in required_model_files if not path.exists()]

if missing_model_files:
    raise FileNotFoundError(
        "以下模型或 Scaler 文件不存在：\n" + "\n".join(missing_model_files)
    )

print("\n正在加载 Scaler...")

scaler_X = joblib.load(SCALER_X_PATH)
scaler_y = joblib.load(SCALER_Y_PATH)

if hasattr(scaler_X, "n_features_in_"):
    if scaler_X.n_features_in_ != len(INPUT_ANGLES):
        raise ValueError(
            f"scaler_X 的输入维度为 "
            f"{scaler_X.n_features_in_}，"
            f"但当前 AVO 模型需要 "
            f"{len(INPUT_ANGLES)} 维输入。"
        )

if hasattr(scaler_y, "n_features_in_"):
    if scaler_y.n_features_in_ != len(TARGET_COLUMNS):
        raise ValueError(
            f"scaler_y 的输出维度为 "
            f"{scaler_y.n_features_in_}，"
            f"但当前模型需要 "
            f"{len(TARGET_COLUMNS)} 维输出。"
        )

print("正在加载模型...")


checkpoint = torch.load(WEIGHT_PATH, map_location=DEVICE, weights_only=True)
if checkpoint["model_config"] != model_config:
    raise ValueError("权重中的模型配置与 Cell 2 不一致，请核对组件和参数。")
if checkpoint["features"] != feature_columns or checkpoint["targets"] != target_columns:
    raise ValueError("权重记录的输入或标签顺序与 Cell 2 不一致。")
model, _ = build_model(**checkpoint["model_config"])
model = model.to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model.eval()

print(f"模型加载完成，计算设备：{DEVICE}")
print(f"输入角度顺序：{INPUT_ANGLES}")
print(f"输出参数顺序：{TARGET_COLUMNS}")
print(f"预测分位数：{QUANTILES_TO_PREDICT}")


def find_time_column(df):
    """查找以 time 开头的时间列；找不到时抛出异常。"""
    for column in df.columns:
        if str(column).lower().startswith("time"):
            return column

    raise ValueError(
        "CSV 中未找到以 time 开头的时间列。\n" f"当前列名：{list(df.columns)}"
    )


def tau_to_tag(tau):
    """将分位数数值转换为可用于文件名的文本标签。"""
    text = f"{float(tau):.10f}".rstrip("0").rstrip(".")

    return text.replace(".", "p")


def index_to_label(index, total_number, tau):
    """三个分位数时按列表位置命名 low、median、upper；其他数量使用数值标签。

    此函数只命名，不检查或调整分位数的数值顺序。
    """
    if total_number == 3:
        return [
            "low",
            "median",
            "upper",
        ][index]

    return f"q{tau_to_tag(tau)}"


def convert_dataframe_to_numeric(
    df,
    file_path,
):
    """将各列转换为数值，无法解析的元素置为 NaN。"""
    df = df.copy()

    for column in df.columns:
        df[column] = pd.to_numeric(
            df[column],
            errors="coerce",
        )

    if df.empty:
        raise ValueError(f"CSV 文件为空：{file_path}")

    return df


def load_profile_input(profile_info):
    """读取三个角度的剖面矩阵，检查时间轴、道名顺序及形状是否一致。"""
    profile_name = profile_info["profile_name"]
    profile_dir = profile_info["profile_dir"]

    input_paths = {
        angle: (profile_dir / f"seismic_theta{angle}.csv") for angle in INPUT_ANGLES
    }

    angle_dataframes = {}
    angle_time_columns = {}

    for angle, input_path in input_paths.items():
        if not input_path.exists():
            raise FileNotFoundError(f"{profile_name} 缺少输入文件：" f"{input_path}")

        df = pd.read_csv(input_path)

        df = convert_dataframe_to_numeric(
            df=df,
            file_path=input_path,
        )

        time_column = find_time_column(df)

        angle_dataframes[angle] = df
        angle_time_columns[angle] = time_column

    reference_angle = INPUT_ANGLES[0]

    reference_df = angle_dataframes[reference_angle]

    reference_time_column = angle_time_columns[reference_angle]

    time_values = reference_df[reference_time_column].to_numpy(dtype=float)

    if not np.isfinite(time_values).all():
        raise ValueError(
            f"{profile_name} 的 theta{reference_angle} " f"时间列中存在 NaN 或 Inf。"
        )

    data_columns = [
        column for column in reference_df.columns if column != reference_time_column
    ]

    if not data_columns:
        raise ValueError(f"{profile_name} 中未检测到地震道数据列。")

    reference_column_set = set(data_columns)

    for angle in INPUT_ANGLES[1:]:
        current_df = angle_dataframes[angle]
        current_time_column = angle_time_columns[angle]

        current_time_values = current_df[current_time_column].to_numpy(dtype=float)

        if len(current_time_values) != len(time_values):
            raise ValueError(
                f"{profile_name} 时间采样数不一致：\n"
                f"theta{reference_angle}："
                f"{len(time_values)}\n"
                f"theta{angle}："
                f"{len(current_time_values)}"
            )

        if not np.allclose(
            current_time_values,
            time_values,
            rtol=0.0,
            atol=1e-8,
            equal_nan=False,
        ):
            raise ValueError(
                f"{profile_name} 的 theta{angle} "
                f"时间轴与 theta{reference_angle} "
                f"不一致。"
            )

        current_data_columns = [
            column for column in current_df.columns if column != current_time_column
        ]

        current_column_set = set(current_data_columns)

        if current_column_set != reference_column_set:
            missing_columns = sorted(reference_column_set - current_column_set)

            extra_columns = sorted(current_column_set - reference_column_set)

            raise ValueError(
                f"{profile_name} 的 theta{angle} "
                f"地震道列不一致。\n"
                f"缺少列：{missing_columns}\n"
                f"多余列：{extra_columns}"
            )

        angle_dataframes[angle] = current_df[
            [current_time_column] + data_columns
        ].copy()

    return (
        time_values,
        data_columns,
        angle_dataframes,
        angle_time_columns,
        reference_time_column,
    )


@torch.no_grad()
def predict_trace_quantiles_avo(seismic_15, seismic_30, seismic_45, taus):
    """一次提取单道特征，返回 (时间采样点数, 分位数数目, 目标数)。

    任一角度为零或非有限值时，该采样点无效；内部缺口先填零再标准化，
    预测完成后恢复为 NaN。两端无效段不送入模型。
    """
    taus = tuple(float(tau) for tau in taus)
    if not taus or any(not 0.0 < tau < 1.0 for tau in taus):
        raise ValueError("taus 必须为非空的 (0,1) 分位数列表。")
    x15 = np.asarray(
        seismic_15,
        dtype=np.float32,
    ).reshape(-1)

    x30 = np.asarray(
        seismic_30,
        dtype=np.float32,
    ).reshape(-1)

    x45 = np.asarray(
        seismic_45,
        dtype=np.float32,
    ).reshape(-1)

    if not (len(x15) == len(x30) == len(x45)):
        raise ValueError("同一道的 theta15、theta30、" "theta45 样本数不一致。")

    x = np.column_stack(
        [
            x15,
            x30,
            x45,
        ]
    ).astype(np.float32)

    number_of_samples = x.shape[0]

    invalid_mask = ~np.isfinite(x).all(axis=1) | (x == 0.0).any(axis=1)

    valid_indices = np.where(~invalid_mask)[0]

    if len(valid_indices) == 0:
        return np.full(
            (number_of_samples, len(taus), len(TARGET_COLUMNS)),
            np.nan,
            dtype=np.float32,
        )

    start_index = valid_indices[0]
    end_index = valid_indices[-1]

    x_valid = x[start_index : end_index + 1].copy()

    invalid_mask_valid = ~np.isfinite(x_valid).all(axis=1) | (x_valid == 0.0).any(
        axis=1
    )

    x_valid[invalid_mask_valid, :] = 0.0

    x_scaled = scaler_X.transform(x_valid)

    x_tensor = torch.tensor(
        x_scaled,
        dtype=torch.float32,
        device=DEVICE,
    )[None, :, :]

    prediction_all = model.predict_quantiles(x_tensor, taus).squeeze(0).cpu().numpy()
    prediction_full = np.full(
        (number_of_samples, len(taus), len(TARGET_COLUMNS)), np.nan, dtype=np.float32
    )
    for qi in range(len(taus)):
        prediction_valid = scaler_y.inverse_transform(prediction_all[:, qi, :])
        prediction_valid[invalid_mask_valid, :] = np.nan
        prediction_full[start_index : end_index + 1, qi, :] = prediction_valid
    return prediction_full


@torch.no_grad()
def predict_trace_avo(seismic_15, seismic_30, seismic_45, tau):
    """保留旧单分位数调用，返回 (时间采样点数, 目标数)。"""
    return predict_trace_quantiles_avo(seismic_15, seismic_30, seismic_45, [tau])[
        :, 0, :
    ]


def update_time_record(
    profile_name,
    prediction_time,
):
    """将本次剖面的耗时写入计时 CSV，更新对应记录。"""
    time_output_path = OUTPUT_ROOT / "Task_times_qhead_AVO.csv"

    current_record = pd.DataFrame(
        [
            {
                "Profile": profile_name,
                "device": DEVICE_TAG,
                "time": round(
                    float(prediction_time),
                    4,
                ),
            }
        ]
    )

    required_columns = [
        "Profile",
        "device",
        "time",
    ]

    if time_output_path.exists():
        old_records = pd.read_csv(time_output_path)

        for column in required_columns:
            if column not in old_records.columns:
                old_records[column] = np.nan

        old_records = old_records[required_columns].copy()

        mask = (old_records["Profile"] == profile_name) & (
            old_records["device"] == DEVICE_TAG
        )

        if mask.any():
            old_records.loc[
                mask,
                "time",
            ] = round(
                float(prediction_time),
                4,
            )
        else:
            old_records = pd.concat(
                [
                    old_records,
                    current_record,
                ],
                ignore_index=True,
            )

        old_records["_profile_number"] = (
            old_records["Profile"]
            .astype(str)
            .str.extract(
                r"profile_s(\d+)",
                expand=False,
            )
        )

        old_records["_profile_number"] = pd.to_numeric(
            old_records["_profile_number"],
            errors="coerce",
        )

        old_records = old_records.sort_values(
            by=[
                "_profile_number",
                "device",
            ],
            na_position="last",
        )

        old_records = old_records.drop(columns=["_profile_number"])

        old_records.to_csv(
            time_output_path,
            index=False,
        )

    else:
        current_record.to_csv(
            time_output_path,
            index=False,
        )

    print(f"任务耗时已更新至：" f"{time_output_path}")


def process_single_profile(profile_info):
    """逐道复用主干预测所有分位数，保持 CSV 命名、时间轴和缺失值约定。"""
    profile_name = profile_info["profile_name"]
    time_values, data_columns, angle_dataframes, _, output_time_column = (
        load_profile_input(profile_info)
    )
    profile_output_dir = OUTPUT_ROOT / profile_name
    profile_output_dir.mkdir(parents=True, exist_ok=True)
    labels = [
        index_to_label(i, len(QUANTILES_TO_PREDICT), float(tau))
        for i, tau in enumerate(QUANTILES_TO_PREDICT)
    ]
    prediction_dictionaries = [
        {target: {output_time_column: time_values} for target in TARGET_COLUMNS}
        for _ in QUANTILES_TO_PREDICT
    ]
    profile_prediction_time = 0.0
    for trace_column in tqdm(data_columns, desc=f"{profile_name} - all quantiles"):
        seismic = [
            angle_dataframes[angle][trace_column].to_numpy(dtype=np.float32)
            for angle in INPUT_ANGLES
        ]
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        start = time.perf_counter()
        prediction = predict_trace_quantiles_avo(*seismic, QUANTILES_TO_PREDICT)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        profile_prediction_time += time.perf_counter() - start
        for qi in range(len(QUANTILES_TO_PREDICT)):
            for target_index, target in enumerate(TARGET_COLUMNS):
                prediction_dictionaries[qi][target][trace_column] = prediction[
                    :, qi, target_index
                ]
    for qi, label in enumerate(labels):
        for target in TARGET_COLUMNS:
            output_path = profile_output_dir / f"{target}_{label}.csv"
            pd.DataFrame(prediction_dictionaries[qi][target]).to_csv(
                output_path, index=False, float_format=CSV_FLOAT_FORMAT
            )
            print(f"输出文件：{output_path}")
    print(f"{profile_name} 多分位数预测处理耗时：{profile_prediction_time:.4f} 秒")
    update_time_record(
        profile_name=profile_name, prediction_time=profile_prediction_time
    )
    return profile_prediction_time


if __name__ == "__main__":
    program_start_time = time.time()

    successful_profiles = []
    failed_profiles = []

    total_pure_prediction_time = 0.0

    for profile_info in PROFILES_TO_PROCESS:
        profile_name = profile_info["profile_name"]

        try:
            current_prediction_time = process_single_profile(profile_info)

            total_pure_prediction_time += current_prediction_time

            successful_profiles.append(profile_name)

        except Exception as error:
            failed_profiles.append(
                {
                    "Profile": profile_name,
                    "Error": str(error),
                }
            )

            print("\n" + "!" * 80)
            print(f"处理 {profile_name} 时发生错误：")
            print(error)
            print("!" * 80)

    total_program_time = time.time() - program_start_time

    print("\n" + "=" * 80)
    print("全部剖面处理结束")
    print("=" * 80)

    print(f"自动识别的有效剖面数：" f"{len(PROFILES_TO_PROCESS)}")

    print(f"成功完成的剖面数：" f"{len(successful_profiles)}")

    if successful_profiles:
        print("成功剖面：" + ", ".join(successful_profiles))

    print(f"全部剖面的纯预测时间：" f"{total_pure_prediction_time:.4f} 秒")

    print(
        f"程序总运行时间"
        f"（含读取、构建表格和写出CSV）："
        f"{total_program_time:.2f} 秒"
    )

    if failed_profiles:
        print("\n以下剖面处理失败：")

        failed_dataframe = pd.DataFrame(failed_profiles)

        print(failed_dataframe.to_string(index=False))

        failed_output_path = OUTPUT_ROOT / "failed_profiles.csv"

        failed_dataframe.to_csv(
            failed_output_path,
            index=False,
        )

        print(f"\n失败信息已保存至：" f"{failed_output_path}")

<a id="workarea-prediction"></a>

## 6. 整个工区预测与结果导出

读取 `workarea/seismic_theta15.csv`、`seismic_theta30.csv`、`seismic_theta45.csv`，逐道预测整个工区，并保留首个角度的时间文本、道名及道顺序。其他角度的时间轴和道名集合必须一致，道顺序自动对齐。

结果保存到 `HCT-QHead/workarea/`，目录不存在时自动创建。输出 Dn、Ip、Is 的 `low`、`median`、`upper` 共 9 个预测 CSV，以及 `Task_times_qhead_AVO.csv`。默认分位数为 0.025、0.5、0.975。

任一角度为零、NaN 或 Inf 的采样点均视为无效，输出恢复为零；两端空值裁剪，内部缺口填零后标准化。预测结果使用磁盘临时矩阵，CSV 按时间行分块导出，结束时清理临时缓存。

已有权重和标准化器时，执行第 1、2 节后可直接运行本节，无需先执行训练、井评价或剖面预测。输入、输出目录和空值、写出设置集中在第 2 节。


In [ ]:
# Cell 6：整个工区逐道预测。已有权重时，执行第 1、2 节后可直接运行。
# 输入、空值规则和磁盘缓存方案参考原版“整个工区”cell。
import tempfile


def workarea_read_angle(csv_path, reference_time=None, reference_columns=None):
    """读取单角度矩阵，保留时间文本，并按首个角度对齐地震道。"""
    header = pd.read_csv(csv_path, nrows=0)
    time_columns = [c for c in header.columns if str(c).strip().lower().startswith("time")]
    if len(time_columns) != 1:
        raise ValueError(f"{csv_path} 应有且仅有一个以 Time 开头的时间列。")
    frame = pd.read_csv(csv_path, dtype={time_columns[0]: "string"}, low_memory=False)
    frame.columns = [str(c).strip() for c in frame.columns]
    if frame.empty or frame.columns.duplicated().any():
        raise ValueError(f"{csv_path} 为空或包含重复列名。")
    time_column = time_columns[0].strip()
    time_text = frame[time_column].fillna("").to_numpy()
    time_values = pd.to_numeric(frame[time_column], errors="coerce").to_numpy(dtype=float)
    if not np.isfinite(time_values).all():
        raise ValueError(f"{csv_path} 的时间轴包含无效值。")
    columns = [c for c in frame.columns if c != time_column]
    if not columns:
        raise ValueError(f"{csv_path} 没有地震道。")
    if reference_time is not None:
        if time_values.shape != reference_time.shape or not np.allclose(
            time_values, reference_time, rtol=0.0, atol=1e-8
        ):
            raise ValueError(f"{csv_path} 的时间轴与首个角度不一致。")
    if reference_columns is not None:
        if set(columns) != set(reference_columns):
            raise ValueError(f"{csv_path} 的地震道列与首个角度不一致。")
        columns = list(reference_columns)
    matrix = frame[columns].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32)
    return time_column, time_text, time_values, columns, matrix


def workarea_load_model():
    """使用当前统一模型接口加载训练权重，核对配置与标准化器。"""
    for path in [MODEL_SAVE_PATH, SCALER_X_PATH, SCALER_Y_PATH]:
        if not path.is_file():
            raise FileNotFoundError(f"缺少模型文件：{path}")
    checkpoint = torch.load(MODEL_SAVE_PATH, map_location=DEVICE, weights_only=True)
    if checkpoint["model_config"] != model_config:
        raise ValueError("权重中的模型配置与第 2 节不一致，请核对组件和参数。")
    if checkpoint["features"] != feature_columns or checkpoint["targets"] != target_columns:
        raise ValueError("权重记录的输入或标签顺序与第 2 节不一致。")
    if [f"Seis{angle}" for angle in INPUT_ANGLES] != feature_columns:
        raise ValueError("INPUT_ANGLES 与训练输入顺序不一致。")
    if list(TARGET_COLUMNS) != target_columns:
        raise ValueError("TARGET_COLUMNS 与训练标签顺序不一致。")
    area_model, _ = build_model(**checkpoint["model_config"])
    area_model = area_model.to(DEVICE)
    area_model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    area_model.eval()
    area_scaler_x = joblib.load(SCALER_X_PATH)
    area_scaler_y = joblib.load(SCALER_Y_PATH)
    for scaler, columns, name in [
        (area_scaler_x, feature_columns, "scaler_X"),
        (area_scaler_y, target_columns, "scaler_y"),
    ]:
        if getattr(scaler, "n_features_in_", None) != len(columns):
            raise ValueError(f"{name} 的维度与模型不一致。")
        if hasattr(scaler, "feature_names_in_") and list(scaler.feature_names_in_) != columns:
            raise ValueError(f"{name} 的列顺序与训练不一致。")
    return area_model, area_scaler_x, area_scaler_y


@torch.inference_mode()
def workarea_predict_trace(trace, area_model, area_scaler_x, area_scaler_y):
    """裁剪两端空值；内部缺口填零后标准化，输出时恢复为空值。"""
    valid = np.isfinite(trace).all(axis=1) & (
        np.abs(trace) > WORKAREA_ZERO_TOLERANCE
    ).all(axis=1)
    indices = np.flatnonzero(valid)
    if indices.size == 0:
        return None
    start, stop = int(indices[0]), int(indices[-1]) + 1
    interval = trace[start:stop].copy()
    invalid = ~valid[start:stop]
    interval[invalid] = 0.0
    scaled = area_scaler_x.transform(interval).astype(np.float32, copy=False)
    tensor = torch.from_numpy(scaled).unsqueeze(0).to(DEVICE)
    labels = ["low", "median", "upper"]
    # 三个分位数共享一次主干计算，使用当前 QHead 的统一推理接口。
    scaled_predictions = area_model.predict_quantiles(
        tensor, QUANTILES_TO_PREDICT
    ).squeeze(0).cpu().numpy()
    if scaled_predictions.shape != (stop - start, len(labels), len(TARGET_COLUMNS)):
        raise ValueError(f"工区分位数预测输出形状错误：{scaled_predictions.shape}")
    predictions = {}
    for index, label in enumerate(labels):
        prediction = area_scaler_y.inverse_transform(scaled_predictions[:, index, :]).astype(
            np.float32, copy=False
        )
        if not np.isfinite(prediction).all():
            raise FloatingPointError("工区预测结果包含 NaN 或 Inf。")
        prediction[invalid] = WORKAREA_EMPTY_OUTPUT_VALUE
        predictions[label] = prediction
    return start, stop, int(valid.sum()), predictions


def workarea_write_csv(output_path, time_column, time_text, columns, prediction_map):
    """按时间行分块导出超宽 CSV，保留时间文本和原始道名顺序。"""
    temporary_path = output_path.with_suffix(".csv.tmp")
    try:
        for start in range(0, len(time_text), WORKAREA_CSV_ROW_CHUNK_SIZE):
            stop = min(start + WORKAREA_CSV_ROW_CHUNK_SIZE, len(time_text))
            block = pd.DataFrame(prediction_map[start:stop], columns=columns, copy=False)
            block.insert(0, time_column, time_text[start:stop])
            block.to_csv(
                temporary_path, mode="w" if start == 0 else "a", header=start == 0,
                index=False, float_format=CSV_FLOAT_FORMAT, encoding="utf-8",
            )
        temporary_path.replace(output_path)
    finally:
        temporary_path.unlink(missing_ok=True)
    print(f"输出文件：{output_path}")


def run_workarea_prediction():
    """加载工区三角度数据，逐道预测并将结果写入当前方法的 workarea 目录。"""
    if not np.isfinite(WORKAREA_ZERO_TOLERANCE) or WORKAREA_ZERO_TOLERANCE < 0:
        raise ValueError("WORKAREA_ZERO_TOLERANCE 必须是非负有限数。")
    if not isinstance(WORKAREA_CSV_ROW_CHUNK_SIZE, int) or WORKAREA_CSV_ROW_CHUNK_SIZE < 1:
        raise ValueError("WORKAREA_CSV_ROW_CHUNK_SIZE 必须为正整数。")
    taus = tuple(float(tau) for tau in QUANTILES_TO_PREDICT)
    if len(taus) != 3 or not 0 < taus[0] < taus[1] == 0.5 < taus[2] < 1:
        raise ValueError("工区预测需要递增的三个分位数，中间为 0.5。")
    labels = ["low", "median", "upper"]
    input_paths = [WORKAREA_INPUT_DIR / f"seismic_theta{angle}.csv" for angle in INPUT_ANGLES]
    for path in input_paths:
        if not path.is_file():
            raise FileNotFoundError(f"缺少工区输入：{path}")
    WORKAREA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    program_start = time.perf_counter()
    area_model, area_scaler_x, area_scaler_y = workarea_load_model()
    print(f"整工区预测：HCT-QHead；设备：{DEVICE}；输出：{WORKAREA_OUTPUT_DIR}")

    matrices = {}
    time_values = columns = None
    for angle, path in zip(INPUT_ANGLES, input_paths):
        print(f"读取：{path}")
        current_time_column, current_time_text, current_times, current_columns, matrix = (
            workarea_read_angle(path, time_values, columns)
        )
        if time_values is None:
            time_column, time_text = current_time_column, current_time_text
            time_values, columns = current_times, current_columns
        matrices[angle] = matrix
    shape = (len(time_values), len(columns))
    cache_bytes = np.prod(shape) * len(TARGET_COLUMNS) * len(labels) * np.dtype(np.float32).itemsize
    print(f"时间采样数：{shape[0]}；地震道数：{shape[1]}")
    print(f"临时预测矩阵预计占用：{cache_bytes / 1024 ** 3:.3f} GB")
    prediction_maps = {}
    prediction_seconds = 0.0
    valid_traces = empty_traces = valid_samples = 0

    # 每次运行使用独立缓存目录；正常结束或发生异常时均释放并清理。
    with tempfile.TemporaryDirectory(prefix="prediction_cache_", dir=WORKAREA_OUTPUT_DIR) as cache_dir:
        try:
            for target in TARGET_COLUMNS:
                for label in labels:
                    prediction_map = np.memmap(
                        Path(cache_dir) / f"{target}_{label}.float32.dat",
                        dtype=np.float32, mode="w+", shape=shape,
                    )
                    prediction_maps[(target, label)] = prediction_map
                    prediction_map[:] = WORKAREA_EMPTY_OUTPUT_VALUE
            for trace_index, trace_column in enumerate(tqdm(columns, desc="HCT-QHead 整工区预测", unit="trace")):
                if DEVICE == "cuda":
                    torch.cuda.synchronize()
                trace_start = time.perf_counter()
                try:
                    trace = np.column_stack([matrices[a][:, trace_index] for a in INPUT_ANGLES])
                    result = workarea_predict_trace(trace, area_model, area_scaler_x, area_scaler_y)
                    if result is None:
                        empty_traces += 1
                        continue
                    start, stop, count, predictions = result
                    for label, prediction in predictions.items():
                        for target_index, target in enumerate(TARGET_COLUMNS):
                            prediction_maps[(target, label)][start:stop, trace_index] = prediction[:, target_index]
                    valid_traces += 1
                    valid_samples += count
                except Exception as error:
                    raise RuntimeError(f"工区地震道 {trace_index} ({trace_column}) 预测失败：{error}") from error
                finally:
                    if DEVICE == "cuda":
                        torch.cuda.synchronize()
                    prediction_seconds += time.perf_counter() - trace_start
            for (target, label), prediction_map in prediction_maps.items():
                prediction_map.flush()
                workarea_write_csv(
                    WORKAREA_OUTPUT_DIR / f"{target}_{label}.csv",
                    time_column, time_text, columns, prediction_map,
                )
        finally:
            for prediction_map in prediction_maps.values():
                prediction_map._mmap.close()
            prediction_maps.clear()
            matrices.clear()

    record = pd.DataFrame([{
        "Area": "workarea", "device": DEVICE_TAG,
        "time_samples": shape[0], "total_traces": shape[1],
        "valid_traces": valid_traces, "empty_traces": empty_traces,
        "valid_samples": valid_samples,
        "pure_prediction_time_seconds": round(prediction_seconds, 4),
        "total_program_time_seconds": round(time.perf_counter() - program_start, 4),
    }])
    time_path = WORKAREA_OUTPUT_DIR / "Task_times_qhead_AVO.csv"
    record.to_csv(time_path, index=False, encoding="utf-8-sig")
    print(f"有效道：{valid_traces}；全空道：{empty_traces}；有效采样点：{valid_samples}")
    print(f"耗时记录：{time_path}")
    print(f"整工区预测完成：{WORKAREA_OUTPUT_DIR.resolve()}")
    return record


workarea_time_record = run_workarea_prediction()
